# 🔬 Prueba de hipótesis: ¿la truncación a 256 tokens daña la detección de riesgo?

**Hipótesis:** los informes largos (que se truncan a 256) son sobre todo de las categorías de riesgo poco frecuentes (BI-RADS 4–6). Ampliar la ventana a 512 podría recuperar señal en esas categorías.

**Evidencia previa (sobre el corpus):** en texto combinado, los informes de riesgo se truncan 20,5% vs. 5,3% en zona segura. En el input del auditor (solo observaciones) el efecto es menor (~0,5% global), pero con solo 11 casos de riesgo en el test, recuperar 1–2 ya movería la sensibilidad.

### Diseño del experimento
- **Experimento principal (válido):** auditor (solo observaciones) a **256 vs 512**. Aísla el efecto de la truncación sobre el modelo real. Comparación interna y controlada (mismo entorno, misma semilla).
- **Diagnóstico opcional (NO es el auditor):** clasificador con texto combinado a 512. ⚠️ El texto combinado **incluye la conclusión, que contiene la categoría BI-RADS**, así que ese modelo *lee* la etiqueta. Su F1 alto no es inferencia ni es comparable con el auditor. Se incluye solo para medir si la truncación afecta al clasificador.

### Instrucciones
1. Activar GPU: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → GPU.
2. Subir `dataset_clean.csv` (celda 3).
3. Ejecutar en orden. Las celdas 7 y 8 son dos entrenamientos (toman tiempo en T4).

## 1 · Instalar y verificar GPU

In [1]:
!pip install -q transformers
import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())
if torch.cuda.is_available(): print("Tarjeta:", torch.cuda.get_device_name(0))
else: print("⚠️ Activa GPU en Entorno de ejecución -> Cambiar tipo de entorno.")

PyTorch: 2.11.0+cu128 | GPU: True
Tarjeta: Tesla T4


## 2 · Librerías y configuración

In [2]:
import numpy as np, pandas as pd, random, re
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, roc_auc_score, confusion_matrix

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
MODELO='PlanTL-GOB-ES/roberta-base-biomedical-clinical-es'; MODELO_ALT='BSC-LT/roberta-base-biomedical-clinical-es'
LR=3e-5; EPOCHS=15; PATIENCE=4; BATCH=8
print("Dispositivo:", DEVICE)

Dispositivo: cuda


## 3 · Subir el dataset

In [3]:
from google.colab import files
subidos=files.upload()
df=pd.read_csv(list(subidos.keys())[0], encoding='utf-8')
print(f"{len(df)} filas. Columnas clave presentes:",
      all(c in df.columns for c in ['auditor_input','texto_input','BI-RADS']))

Saving dataset_clean.csv to dataset_clean.csv
4357 filas. Columnas clave presentes: True


## 4 · Diagnóstico de la hipótesis (longitud por categoría)

In [4]:
try: tokd=AutoTokenizer.from_pretrained(MODELO)
except Exception: MODELO=MODELO_ALT; tokd=AutoTokenizer.from_pretrained(MODELO)
def wc(s): return len(re.findall(r'\w+',str(s),flags=re.UNICODE))
ratio=(df['n_tokens']/df['texto_input'].apply(wc)).median() if 'n_tokens' in df.columns else 1.81
df['tok_aud_est']=df['auditor_input'].apply(wc)*ratio
print("Tokens del auditor (estimado) por zona:")
for z,nombre in [(df['BI-RADS']<4,'SEGURA 0-3'),(df['BI-RADS']>=4,'RIESGO 4-6')]:
    sub=df[z]['tok_aud_est']; print(f"  {nombre}: mediana {int(sub.median())} | máx {int(sub.max())} | % >256: {100*(sub>256).mean():.1f}%")

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/540k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Tokens del auditor (estimado) por zona:
  SEGURA 0-3: mediana 128 | máx 310 | % >256: 0.5%
  RIESGO 4-6: mediana 164 | máx 312 | % >256: 2.7%


## 5 · Funciones: split, augmentación, dataset, modelo y entrenamiento

In [5]:
def aumentar(texto):
    V=[('bordes irregulares','márgenes irregulares'),('bordes espiculados','márgenes espiculados'),
       ('bordes mal definidos','límites imprecisos'),('imagen nodular','nódulo'),('nódulo','imagen nodular'),
       ('lesión nodular','nódulo'),('heterogéneamente densas','de densidad heterogénea'),
       ('muy densas','extremadamente densas'),('calcificaciones sospechosas','depósitos cálcicos sospechosos'),
       ('microcalcificaciones','calcificaciones puntiformes agrupadas'),('mama derecha','hemimama derecha'),
       ('mama izquierda','hemimama izquierda'),('se visualiza','se observa'),('se observa','se visualiza'),
       ('se evidencia','se observa')]
    t=texto
    for o,r in random.sample(V,min(3,len(V))):
        if o in t: t=t.replace(o,r,1)
    return t

def construir_splits(columna):
    X=df[columna].values; y=df['BI-RADS'].values
    Xtv,Xte,ytv,yte=train_test_split(X,y,test_size=0.15,random_state=SEED,stratify=y)
    Xtr,Xva,ytr,yva=train_test_split(Xtv,ytv,test_size=0.176,random_state=SEED,stratify=ytv)
    mask=np.isin(ytr,[3,4,5,6]); ta,la=[],[]
    for txt,lab in zip(Xtr[mask],ytr[mask]):
        for _ in range(3): ta.append(aumentar(txt)); la.append(lab)
    Xtra=np.concatenate([Xtr,np.array(ta)]); ytra=np.concatenate([ytr,np.array(la)])
    idx=np.random.RandomState(SEED).permutation(len(Xtra))
    return Xtra[idx],ytra[idx],Xva,yva,Xte,yte

class DS(Dataset):
    def __init__(self,t,l,tok,ml): self.t=list(t); self.l=list(l); self.tok=tok; self.ml=ml
    def __len__(self): return len(self.t)
    def __getitem__(self,i):
        e=self.tok(str(self.t[i]),truncation=True,padding='max_length',max_length=self.ml,
                   return_tensors='pt',return_token_type_ids=False)
        return {'input_ids':e['input_ids'].squeeze(0),'attention_mask':e['attention_mask'].squeeze(0),
                'labels':torch.tensor(self.l[i],dtype=torch.long)}

class Modelo(nn.Module):
    def __init__(self,m,n=7,dp=0.5):
        super().__init__(); self.enc=AutoModel.from_pretrained(m)
        self.dp=nn.Dropout(dp); self.cl=nn.Linear(self.enc.config.hidden_size,n)
    def forward(self,ids,mask):
        return self.cl(self.dp(self.enc(input_ids=ids,attention_mask=mask).last_hidden_state[:,0,:]))

def entrenar_evaluar(columna, max_length, etiqueta):
    print(f"\n{'='*60}\n  {etiqueta}  (columna={columna}, max_length={max_length})\n{'='*60}")
    Xtra,ytra,Xva,yva,Xte,yte=construir_splits(columna)
    tok=AutoTokenizer.from_pretrained(MODELO)
    tl=DataLoader(DS(Xtra,ytra,tok,max_length),batch_size=BATCH,shuffle=True)
    vl=DataLoader(DS(Xva,yva,tok,max_length),batch_size=BATCH)
    tt=DataLoader(DS(Xte,yte,tok,max_length),batch_size=BATCH)
    clases=np.unique(ytra); pw=compute_class_weight('balanced',classes=clases,y=ytra)
    pv=np.ones(7,dtype=np.float32)
    for c,w in zip(clases,pw): pv[c]=w
    crit=nn.CrossEntropyLoss(weight=torch.tensor(pv).to(DEVICE))
    model=Modelo(MODELO).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR)
    sch=get_linear_schedule_with_warmup(opt,0,len(tl)*EPOCHS)
    amp=(DEVICE=='cuda'); scaler=torch.cuda.amp.GradScaler(enabled=amp)
    def ep(loader,train=True):
        model.train() if train else model.eval(); tot=0; P=[]; R=[]
        with torch.set_grad_enabled(train):
            for j,b in enumerate(loader):
                ids=b['input_ids'].to(DEVICE); mk=b['attention_mask'].to(DEVICE); lb=b['labels'].to(DEVICE)
                if train: opt.zero_grad()
                with torch.cuda.amp.autocast(enabled=amp):
                    lo=model(ids,mk); loss=crit(lo,lb)
                if train: scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
                tot+=loss.item(); P.extend(lo.argmax(1).cpu().numpy()); R.extend(lb.cpu().numpy())
                if train and (j+1)%150==0: print(f"      batch {j+1}/{len(loader)}",flush=True)
        return tot/len(loader), f1_score(R,P,average='macro',zero_division=0)
    mejor,sin=0,0
    for e in range(1,EPOCHS+1):
        trl,trf=ep(tl,True); vll,vlf=ep(vl,False); m=""
        if vlf>mejor: mejor,sin=vlf,0; torch.save(model.state_dict(),'best_tmp.pt'); m="  <-"
        else: sin+=1
        print(f"  Época {e:2d} | TrainF1 {trf:.4f} | ValF1 {vlf:.4f}{m}",flush=True)
        if sin>=PATIENCE: print(f"  Early stop en época {e}"); break
    model.load_state_dict(torch.load('best_tmp.pt',map_location=DEVICE)); model.eval()
    preds=[]; probs=[]; reales=[]
    with torch.no_grad():
        for b in tt:
            ids=b['input_ids'].to(DEVICE); mk=b['attention_mask'].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=amp): lo=model(ids,mk)
            pr=torch.softmax(lo.float(),1).cpu().numpy()
            probs.extend(pr); preds.extend(pr.argmax(1)); reales.extend(b['labels'].numpy())
    preds=np.array(preds); reales=np.array(reales); probs=np.array(probs)
    f1m=f1_score(reales,preds,average='macro',zero_division=0)
    pr=probs[:,4:7].sum(1); yz=(reales>=4).astype(int); auc=roc_auc_score(yz,pr)
    yp=(pr>=0.5).astype(int); tn,fp,fn,tp=confusion_matrix(yz,yp,labels=[0,1]).ravel()
    sens=tp/(tp+fn) if (tp+fn) else 0; espec=tn/(tn+fp) if (tn+fp) else 0
    print(f"  >> F1-macro: {f1m:.4f} | AUC zona: {auc:.4f} | Sens: {sens:.3f} | Espec: {espec:.3f}")
    return {'f1':f1m,'auc':auc,'sens':sens,'espec':espec,'preds':preds,'reales':reales,'probs':probs}

## 6 · Experimento principal — Auditor a 256 (baseline controlado)

In [6]:
r256 = entrenar_evaluar('auditor_input', 256, 'AUDITOR @ 256 (baseline)')


  AUDITOR @ 256 (baseline)  (columna=auditor_input, max_length=256)


pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_689/3200325955.py:55: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp=(DEVICE=='cuda'); scaler

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

...

      batch 150/424
      batch 300/424


/tmp/ipykernel_689/3200325955.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


  Época  9 | TrainF1 0.9820 | ValF1 0.5063
  Early stop en época 9


/tmp/ipykernel_689/3200325955.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp): lo=model(ids,mk)


  >> F1-macro: 0.6017 | AUC zona: 0.8402 | Sens: 0.636 | Espec: 0.991


## 7 · Experimento principal — Auditor a 512 (la hipótesis)

In [7]:
r512 = entrenar_evaluar('auditor_input', 512, 'AUDITOR @ 512 (hipótesis)')


  AUDITOR @ 512 (hipótesis)  (columna=auditor_input, max_length=512)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_689/3200325955.py:55: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp=(DEVICE=='cuda'); scaler

      batch 150/424
      batch 300/424


/tmp/ipykernel_689/3200325955.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


...

      batch 150/424
      batch 300/424


/tmp/ipykernel_689/3200325955.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


  Época  7 | TrainF1 0.9723 | ValF1 0.4645
  Early stop en época 7


/tmp/ipykernel_689/3200325955.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp): lo=model(ids,mk)


  >> F1-macro: 0.4780 | AUC zona: 0.9580 | Sens: 0.545 | Espec: 0.992


## 8 · Comparación 256 vs 512 y casos de riesgo largos

In [8]:
print("="*58)
print("RESULTADO DE LA HIPÓTESIS (auditor, solo observaciones)")
print("="*58)
print(f"{'':12} | {'F1-macro':>9} | {'AUC':>6} | {'Sens':>6} | {'Espec':>6}")
print(f"{'256 tokens':12} | {r256['f1']:>9.4f} | {r256['auc']:>6.3f} | {r256['sens']:>6.3f} | {r256['espec']:>6.3f}")
print(f"{'512 tokens':12} | {r512['f1']:>9.4f} | {r512['auc']:>6.3f} | {r512['sens']:>6.3f} | {r512['espec']:>6.3f}")
d_f1=r512['f1']-r256['f1']; d_auc=r512['auc']-r256['auc']; d_se=r512['sens']-r256['sens']
print(f"{'Δ (512-256)':12} | {d_f1:>+9.4f} | {d_auc:>+6.3f} | {d_se:>+6.3f} |")

# Foco en los casos de RIESGO largos (los que a 256 se truncaban)
Xtra,ytra,Xva,yva,Xte,yte=construir_splits('auditor_input')
tok=AutoTokenizer.from_pretrained(MODELO)
largos_test=np.array([len(tok(str(t),truncation=False)['input_ids'])>256 for t in Xte])
riesgo_test=(yte>=4)
foco=riesgo_test & largos_test
print(f"\nCasos de RIESGO largos (>256 tok) en el test: {int(foco.sum())}")
if foco.sum()>0:
    ok256=((r256['preds']>=4)[foco]).sum(); ok512=((r512['preds']>=4)[foco]).sum()
    print(f"  Detectados como riesgo a 256: {ok256}/{int(foco.sum())}")
    print(f"  Detectados como riesgo a 512: {ok512}/{int(foco.sum())}")
    print("  -> Si 512 detecta más, la hipótesis se sostiene en estos casos.")
else:
    print("  No hay casos de riesgo largos en el test (efecto nulo esperable).")

RESULTADO DE LA HIPÓTESIS (auditor, solo observaciones)
             |  F1-macro |    AUC |   Sens |  Espec
256 tokens   |    0.6017 |  0.840 |  0.636 |  0.991
512 tokens   |    0.4780 |  0.958 |  0.545 |  0.992
Δ (512-256)  |   -0.1237 | +0.118 | -0.091 |

Casos de RIESGO largos (>256 tok) en el test: 0
  No hay casos de riesgo largos en el test (efecto nulo esperable).


## 9 · (Opcional) Diagnóstico: clasificador con texto combinado a 512

⚠️ **Esto NO es el auditor.** El texto combinado incluye la conclusión, que contiene la categoría BI-RADS. El modelo *lee* la etiqueta, por eso su F1 será alto (~0,90). No es inferencia, no es comparable con el auditor, y **no debe reportarse como resultado del auditor**. Sirve solo para ver si la truncación afecta al clasificador. Ejecuta esta celda solo si quieres ese dato.

In [9]:
rc512 = entrenar_evaluar('texto_input', 512, 'CLASIFICADOR @ 512 (lee etiqueta, NO es el auditor)')
print(f"\nNota: baseline clasificador a 256 = F1-macro ~0,9027. Comparar solo a título diagnóstico.")


  CLASIFICADOR @ 512 (lee etiqueta, NO es el auditor)  (columna=texto_input, max_length=512)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_689/3200325955.py:55: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp=(DEVICE=='cuda'); scaler

      batch 150/424
      batch 300/424


/tmp/ipykernel_689/3200325955.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


...

      batch 150/424
      batch 300/424


/tmp/ipykernel_689/3200325955.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):


  Época  7 | TrainF1 0.9852 | ValF1 0.9519
  Early stop en época 7


/tmp/ipykernel_689/3200325955.py:80: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp): lo=model(ids,mk)


  >> F1-macro: 0.9990 | AUC zona: 1.0000 | Sens: 1.000 | Espec: 1.000

Nota: baseline clasificador a 256 = F1-macro ~0,9027. Comparar solo a título diagnóstico.


## 10 · Cómo interpretar

- **Mira primero la celda 8.** Si a 512 sube la sensibilidad o el AUC respecto a 256, y los casos de riesgo largos se detectan mejor, tu hipótesis se sostiene y conviene adoptar 512 (o 384) para el auditor.
- **Si no cambia**, confirma que truncar a 256 no perdía señal relevante en las observaciones, y te quedas en 256 por costo. También es un resultado válido y publicable.
- La celda 9 (clasificador) es solo diagnóstica. No la reportes como auditor.

Si 512 ayuda, descarga el modelo y registra el experimento en la rama `feat/512-tokens`.